# Huren vs. Kopen — Cumulatieve kosten over de tijd

Vergelijking van de cumulatieve woonlasten bij **huren** versus **kopen** van de woning aan de Willebrordusstraat 135B-04, Rotterdam.

De aannames staan als parameters bovenaan, zodat je makkelijk scenario's kunt doorrekenen.

**Belangrijke conceptuele keuze:** bij kopen tellen we alleen de *werkelijke kosten* mee (rente + heffingen + VvE), **niet** de aflossing — want aflossing is vermogensopbouw, geen verloren geld. Aan het eind verrekenen we de opbouw (aflossing + waardestijging − verkoopkosten − kosten tweede huis) als je zou verkopen en doorverhuizen.


In [1]:
import numpy as np
import matplotlib.pyplot as plt


## 1. Aannames (parameters)

Pas deze waarden aan om scenario's door te rekenen.

In [ ]:
# ---- HUUR ----
HUUR_NU            = 1857.0     # huidige kale huur per maand (euro)
HUUR_STIJGING      = 0.04       # jaarlijkse huurverhoging (4%)

# ---- KOOP: hypotheek ----
# Je financiert 100%, dus koopsom = hypotheekbedrag.
KOOPSOM            = 480000.0   # koopsom / hypotheekbedrag
RENTE              = 0.0422     # hypotheekrente (boven NHG-grens)
LOOPTIJD           = 30         # looptijd in jaren (vast)

# Rente + aflossing per jaar volgen uit de annuïteitenformule (zie functies).
# Het fiscaal voordeel volgt uit een lineair model dat op beide V&W-doorrekeningen
# (470k @ 3,89% en 480k @ 4,22%) past:
#     fiscaal(jaar) = (rente(jaar) - EWF_JAAR) * AFTREK_TARIEF
EWF_JAAR           = 1215       # eigenwoningforfait per jaar (uit doorrekening)
AFTREK_TARIEF      = 0.4395     # gehanteerd aftrektarief (uit doorrekening)

# ---- Voorwaardelijke regels op de koopsom ----
NHG_GRENS          = 470000     # onder dit bedrag: NHG van toepassing
NHG_RENTE          = 0.0389     # rente met NHG
NHG_KOSTEN         = 1880       # eenmalige NHG-kosten
STARTERS_GRENS     = 555000     # boven dit bedrag vervalt de startersvrijstelling
OVERDRACHTSBELASTING = 0.02     # overdrachtsbelasting (2%) boven de startersgrens

# ---- KOOP: lokale heffingen + VvE ----
OZB_TARIEF         = 0.000643
RIOOL_PER_MND      = 27.0
VVE_PER_MND        = 200.0      # SCHATTING - opvragen bij VvE
ORV_PER_MND        = 15.0

# ---- JAARLIJKSE STIJGING ----
WOZ_STIJGING       = 0.04
HEFFING_STIJGING   = 0.03

# ---- EENMALIGE KOSTEN ----
AANKOOPKOSTEN_NU   = 6552.0     # kosten koper: notaris, taxatie, hypotheekadvies
EXTRA_KOSTEN_KOPER = 0.0        # aanvullende eenmalige kosten, bijv. bouwkundige keuring
OVERBIEDEN         = 0.0        # marktwaarde - koopsom (extra vermogen bij verkoop)

# ---- WAARDEONTWIKKELING & VERKOOP ----
WAARDESTIJGING     = 0.00
MAKELAAR_PCT       = 0.0125

# ---- OVERSTAP NAAR TWEEDE HUIS ----
OVERSTAP_MEEREKENEN = True
TWEEDE_HUIS_PRIJS   = 600000
OVERDRACHTSBEL_PCT  = 0.02
NIEUWE_FIN_KOSTEN   = 6552.0

HORIZON_JAREN      = 5


## 2. Hulpfuncties

We berekenen voor elk jaar de cumulatieve *kosten* (verloren geld) van huren en kopen.

In [ ]:
def effectieve_rente():
    """Onder de NHG-grens geldt automatisch het NHG-tarief."""
    return NHG_RENTE if KOOPSOM < NHG_GRENS else RENTE


def jaar_schema():
    """Annuïteitenschema: rente en aflossing per jaar uit (KOOPSOM, rente, LOOPTIJD)."""
    i = effectieve_rente() / 12
    n = LOOPTIJD * 12
    M = KOOPSOM * i / (1 - (1 + i) ** -n)
    rente, aflossing = {}, {}
    bal = KOOPSOM
    for m in range(1, n + 1):
        r = bal * i
        a = M - r
        bal -= a
        j = (m - 1) // 12 + 1
        rente[j] = rente.get(j, 0) + r
        aflossing[j] = aflossing.get(j, 0) + a
    return rente, aflossing


def fiscaal_jaar(rente_jaar):
    """Hypotheekrenteaftrek: (rente - eigenwoningforfait) * aftrektarief."""
    return (rente_jaar - EWF_JAAR) * AFTREK_TARIEF


def eenmalige_kosten():
    """Kosten koper + extra + (NHG-kosten of overdrachtsbelasting)."""
    totaal = AANKOOPKOSTEN_NU + EXTRA_KOSTEN_KOPER
    if KOOPSOM > STARTERS_GRENS:
        totaal += KOOPSOM * OVERDRACHTSBELASTING
    if KOOPSOM < NHG_GRENS:
        totaal += NHG_KOSTEN
    return totaal


def huur_kosten_cumulatief(jaren):
    """Cumulatieve huurkosten; huur stijgt elk jaar met HUUR_STIJGING."""
    cum, totaal = [], 0.0
    for j in range(1, jaren + 1):
        totaal += HUUR_NU * 12 * (1 + HUUR_STIJGING) ** (j - 1)
        cum.append(totaal)
    return np.array(cum)


def heffingen_jaar(j):
    """Jaarlijkse heffingen: OZB (groeit met WOZ) + VvE/riool/ORV (groeit met heffingenstijging)."""
    woz = KOOPSOM * (1 + WOZ_STIJGING) ** (j - 1)
    ozb = woz * OZB_TARIEF
    overige_mnd = (RIOOL_PER_MND + VVE_PER_MND + ORV_PER_MND) * (1 + HEFFING_STIJGING) ** (j - 1)
    return ozb + overige_mnd * 12


def koop_kosten_cumulatief(jaren, schema):
    """Cumulatieve KOSTEN van kopen (rente + heffingen - fiscaal voordeel).
    Aflossing telt NIET mee (dat is vermogensopbouw)."""
    rente, _ = schema
    cum, totaal = [], eenmalige_kosten()
    for j in range(1, jaren + 1):
        r = rente.get(j, 0)
        totaal += r + heffingen_jaar(j) - fiscaal_jaar(r)
        cum.append(totaal)
    return np.array(cum)


def overstapkosten_tweede_huis(woningwaarde_nu):
    """Eenmalige kosten van het kopen van een volgend huis na verkoop."""
    if not OVERSTAP_MEEREKENEN:
        return 0.0
    prijs_2e = TWEEDE_HUIS_PRIJS if TWEEDE_HUIS_PRIJS is not None else woningwaarde_nu
    return prijs_2e * OVERDRACHTSBEL_PCT + NIEUWE_FIN_KOSTEN


def opgebouwd_vermogen(jaren, schema):
    """Netto vermogen bij verkoop in jaar j.
    startwaarde = KOOPSOM + OVERBIEDEN groeit mee met WAARDESTIJGING."""
    _, aflossing = schema
    cum, afgelost = [], 0.0
    marktwaarde_start = KOOPSOM + OVERBIEDEN
    for j in range(1, jaren + 1):
        afgelost += aflossing.get(j, 0)
        woningwaarde = marktwaarde_start * (1 + WAARDESTIJGING) ** j
        waardestijging = woningwaarde - KOOPSOM
        verkoopkosten = woningwaarde * MAKELAAR_PCT
        overstap = overstapkosten_tweede_huis(woningwaarde)
        cum.append(afgelost + waardestijging - verkoopkosten - overstap)
    return np.array(cum)


## 3. Bereken en plot

We tonen drie lijnen:
- **Huur cumulatief** — al het huurgeld dat je kwijt bent.
- **Koop cumulatieve kosten** — rente + heffingen − fiscaal voordeel + eenmalige kosten (geen aflossing).
- **Koop netto kosten** — koopkosten *minus* opgebouwd vermogen (aflossing + waardestijging − makelaarscourtage − kosten tweede huis); dit is wat kopen je "echt" kost als je op dat moment zou verkopen en doorverhuizen.

In [ ]:
jaren = np.arange(1, HORIZON_JAREN + 1)

schema     = jaar_schema()
huur_cum   = huur_kosten_cumulatief(HORIZON_JAREN)
koop_cum   = koop_kosten_cumulatief(HORIZON_JAREN, schema)
vermogen   = opgebouwd_vermogen(HORIZON_JAREN, schema)
koop_netto = koop_cum - vermogen

fig, ax = plt.subplots(figsize=(6.5, 6.5))

ax.plot(jaren, huur_cum,   marker='o', lw=2.2, label='Huur (cumulatief verloren)')
ax.plot(jaren, koop_netto, marker='^', lw=2.2, label='Koop: netto kosten (na vermogensopbouw)')

ax.axhline(0, color='grey', lw=0.8, ls='--')

diff = huur_cum - koop_netto
for i in range(len(diff) - 1):
    if diff[i] < 0 and diff[i+1] >= 0:
        ax.axvline(jaren[i+1], color='green', ls=':', alpha=0.6)
        ax.text(jaren[i+1], ax.get_ylim()[1]*0.05,
                f'  break-even ~jaar {jaren[i+1]}', color='green', fontsize=9)
        break

ax.set_xlabel('Jaren', fontsize=12)
ax.set_ylabel('Cumulatief bedrag (euro)', fontsize=12)
ax.set_title('Huren vs. Kopen — cumulatieve kosten over de tijd\nRotterdam', fontsize=13)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'EUR {x:,.0f}'))

plt.tight_layout()
plt.savefig('huur_vs_koop.png', dpi=130, bbox_inches='tight')
plt.show()


## 4. Tabel met getallen per jaar

In [67]:
print(f"{'Jaar':>4} | {'Huur cum.':>12} | {'Koop kosten':>12} | {'Koop netto':>12} | {'Verschil':>12}")
print('-' * 66)
for i, j in enumerate(jaren):
    verschil = huur_cum[i] - koop_netto[i]   # positief = kopen voordeliger
    print(f"{j:>4} | {huur_cum[i]:>12,.0f} | {koop_cum[i]:>12,.0f} | "
          f"{koop_netto[i]:>12,.0f} | {verschil:>+12,.0f}")
print('-' * 66)
print("Verschil positief = kopen is op dat moment voordeliger dan huren.")


Jaar |    Huur cum. |  Koop kosten |   Koop netto |     Verschil
------------------------------------------------------------------
   1 |       22,284 |       15,688 |       36,215 |      -13,931
   2 |       45,459 |       30,703 |       42,922 |       +2,538
   3 |       69,562 |       45,619 |       49,173 |      +20,388
   4 |       94,628 |       60,434 |       54,950 |      +39,678
   5 |      120,697 |       75,140 |       60,229 |      +60,468
------------------------------------------------------------------
Verschil positief = kopen is op dat moment voordeliger dan huren.


## 5. Scenario's spelen

Wijzig hierboven bijvoorbeeld:
- `WAARDESTIJGING = 0.0` — wat als de woning niet in waarde stijgt?
- `WOZ_STIJGING` — hoe hard de WOZ-waarde (en dus de OZB) jaarlijks oploopt.
- `HEFFING_STIJGING` — jaarlijkse stijging van riool, VvE en ORV.
- `VVE_PER_MND` — de grootste onbekende; vul het echte bedrag in zodra je het servicekostenoverzicht hebt.
- `HUUR_STIJGING` — bij een hogere huurverhoging wordt kopen sneller voordelig.
- `OVERSTAP_MEEREKENEN = False` — laat de kosten van een tweede huis (overdrachtsbelasting + financieringskosten) buiten beschouwing.
- `TWEEDE_HUIS_PRIJS` — vul een concrete koopsom in voor het volgende huis (None = even duur als deze woning op het verkoopmoment).

Run daarna de cellen opnieuw om de plot en tabel bij te werken.